# 💸 Notebook 1: The Double-Charge Bug

**Goal:** see *why* naive APIs double-charge customers on retries.

Networks are unreliable. A client sends `POST /charge`. The server processes it and sends a `200 OK`. The response packet is lost. The client sees a timeout and retries. The server — with no memory of the first call — charges again.

Most payment / messaging / ordering bugs trace back to this pattern.

## 🧭 Quick primer: which HTTP methods are idempotent?

An operation is **idempotent** if calling it once has the same effect as calling it N times.

| Method   | Idempotent by spec? | Why |
|---------:|:-------------------:|-----|
| `GET`    | ✅ | just reads data |
| `PUT`    | ✅ | replaces the whole resource — doing it twice leaves the same state |
| `DELETE` | ✅ | deleting something that's already gone is a no-op |
| `POST`   | ❌ | each call usually creates a *new* resource / side-effect |

`POST /charge` is the dangerous one. It's our subject for the rest of this lab.

## 🛠️ Setup

```bash
cd 04-patterns/idempotency
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟥 BAD: server keeps no memory of requests

We'll simulate a flaky network: the server's response is lost some fraction of the time, so the client retries.

In [1]:
import random
random.seed(1)

balances = {'alice': 100}

def server_charge(account, amount):
    # side-effect: money moves
    balances[account] -= amount
    return {'ok': True, 'balance': balances[account]}

def flaky_network_call(account, amount, drop_response_prob=0.6):
    """Server ALWAYS processes the request. But the response is sometimes lost."""
    result = server_charge(account, amount)
    if random.random() < drop_response_prob:
        raise TimeoutError('response lost in the network')
    return result

def client_charge_with_retry(account, amount, max_attempts=5):
    for attempt in range(1, max_attempts + 1):
        try:
            r = flaky_network_call(account, amount)
            print(f'  attempt {attempt}: ✅ got response {r}')
            return r
        except TimeoutError:
            print(f'  attempt {attempt}: ⏱️ timeout — retrying')
    raise RuntimeError('gave up')

print('Client asked to charge $10 once:')
client_charge_with_retry('alice', 10)
print(f"\nAlice's balance ended at {balances['alice']} — the client intended ONE charge of $10.")
print(f"Every silent retry became a real charge. 😱")


Client asked to charge $10 once:
  attempt 1: ⏱️ timeout — retrying
  attempt 2: ✅ got response {'ok': True, 'balance': 80}

Alice's balance ended at 80 — the client intended ONE charge of $10.
Every silent retry became a real charge. 😱


### 🔍 What went wrong?

- The *request* arrived at the server successfully every time.
- Only the *response* was lost.
- The server has no way to tell "this is a retry of the previous request" apart from "this is a brand-new charge".

👉 Next notebook: the client attaches an **idempotency key** so the server *can* tell them apart.